# NRAT Scraper — сбор PDF по годам (Google Colab)

Собирает PDF c nrat.ukrintei.ua. Один запуск = один год (задаётся в ячейке «Настройки», `YEAR`).
Файлы сохраняются на Google Drive в отдельных папках по годам: `MyDrive/nrat_pdfs/<год>/<год-месяц>/<день>/`.

## Порядок
1. **Ячейки 1–4** (Установка → Настройки → Функции → Логика) — запусти по очереди. На ячейке 1 разреши доступ к Google Drive.
2. **СОБРАТЬ ВЕСЬ ГОД** — большой сбор на часы. Если Colab оборвался — снова запусти ячейки 1–4 и эту: продолжит с места обрыва.

Чтобы собрать следующий год — поменяй `YEAR` в ячейке 2 и пройди шаги заново.
Готовые файлы лежат прямо на Google Drive (`nrat_pdfs/<год>/`) — отдельный архив не нужен.

⚠️ Паузы между запросами не уменьшать — иначе бан на 1–2 дня.

In [ ]:
# ==============================================
# ЯЧЕЙКА 1 — установка, импорты, подключение Google Drive
# ==============================================
!pip install requests beautifulsoup4 tqdm -q

import requests
from bs4 import BeautifulSoup
import os
import time
import re
import json
from datetime import datetime, timedelta
from urllib.parse import urlencode
from tqdm import tqdm

# --- Подключаем Google Drive, чтобы НИЧЕГО не терялось при обрыве сессии ---
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключён")

In [ ]:
# ==============================================
# ЯЧЕЙКА 2 — КОНФИГУРАЦИЯ
# ==============================================

# >>> ГЛАВНОЕ: какой год собираем СЕЙЧАС. Меняй вручную: 2025, потом 2024, 2023 ... <<<
# Один запуск = один год. Чтобы собрать следующий — поменяй YEAR и запусти заново.
YEAR = 2025

# Куда сохраняем (прямо на Google Drive, чтобы пережить обрыв сессии).
# У каждого года — своя отдельная папка: nrat_pdfs/<год>/<год-месяц>/<день>/
DOWNLOAD_FOLDER = "/content/drive/MyDrive/nrat_pdfs"

# Чекпоинт у каждого года свой -> чистое продолжение после обрыва
CHECKPOINT_FILE = os.path.join(DOWNLOAD_FOLDER, f"_progress_{YEAR}.json")

# Поиск
BASE_SEARCH_URL = "https://nrat.ukrintei.ua/searchdb/?"
BASE_PARAMS = {
    '_token': '6C0elymE1XyE8AOayLEsuYco3JewHUAF0DazKx9Y',  # обновляется автоматически в ячейке 3
    'typeSearch2': 'ok',
    'typeCategory[]': '0',
    'lcSource': '',
    'authorSearch': '',
    'specialnistSearch[]': '0',
    'temaSearch2': '',
    'textSearch': '',
    'registrationNumberSearch': '',
    'firm_id': '0',
    'sortOrder': 'registration_date',
    'sortDir': 'desc',
    'tab': 'big'
}

DAYS_PER_CHUNK = 1   # 1 день за запрос — чтобы укладываться в лимит сайта (~1000 рез.)

# --- ПАУЗЫ (НЕ УМЕНЬШАТЬ! иначе бан на 1-2 дня) ---
DELAY_BETWEEN_PAGES = 3   # сек между страницами выдачи
DELAY_BETWEEN_FILES = 1   # сек между скачиванием PDF
DELAY_BETWEEN_DAYS  = 5   # сек между днями

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)


def day_folder_for(year, date_from):
    """
    Строит путь к папке для конкретного дня на Google Drive.

    Структура: nrat_pdfs/<год>/<год-месяц>/<дата>/
    Например: nrat_pdfs/2025/2025-01/2025-01-15/

    Параметры:
        year      — числовой год (int), например 2025
        date_from — строка даты в формате 'YYYY-MM-DD'

    Возвращает: полный путь к папке дня (строка)
    """
    return os.path.join(DOWNLOAD_FOLDER, str(year), date_from[:7], date_from)


# Сессия с заголовками браузера
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,uk;q=0.8',
})

print(f"Собираем год: {YEAR}")
print("Папка на Drive:", os.path.join(DOWNLOAD_FOLDER, str(YEAR)))

In [ ]:
# ==============================================
# ЯЧЕЙКА 3 — ФУНКЦИИ
# ==============================================
# Эта ячейка ничего не качает — она только ОПРЕДЕЛЯЕТ функции (заготовки команд).
# Реально они вызываются в ячейке 4 и в ячейке «СОБРАТЬ ВЕСЬ ГОД».


# ---------- ЧЕКПОИНТ (продолжение после обрыва) ----------

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):                       # if = «если». os.path.exists проверяет, есть ли файл чекпоинта на Drive
        try:                                                  # try = «попробовать»: код ниже может упасть (файл битый) — ошибку поймаем
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:   # open(...,'r') открывает файл на ЧТЕНИЕ; as f даёт ему короткое имя f; with сам закроет файл
                return set(json.load(f).get('completed_dates', []))   # json.load(f) читает JSON; .get берёт список дат (или [] если ключа нет); set(...) превращает в множество; return = «вернуть результат»
        except Exception:                                     # except = «если внутри try случилась ЛЮБАЯ ошибка»
            return set()                                      # вернуть пустое множество (считаем, что готовых дней нет)
    return set()                                              # файла нет вообще — тоже вернуть пустое множество


def save_checkpoint(completed_set):                           # completed_set — входной параметр: множество завершённых дат
    try:                                                      # пробуем записать; если Drive недоступен — не роняем весь сбор
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:   # open(...,'w') открывает файл на ЗАПИСЬ (старое содержимое перезатирается)
            json.dump({'completed_dates': sorted(completed_set)}, f, ensure_ascii=False, indent=2)   # json.dump пишет словарь в файл; sorted сортирует даты; ensure_ascii=False — кириллицу не ломать; indent=2 — красивые отступы
    except Exception as e:                                    # as e = «положить объект ошибки в переменную e», чтобы показать её текст
        print(f"  ⚠️ Не удалось записать чекпоинт: {e}")      # f"..." — f-строка: всё в {...} подставляется как значение; здесь {e} = текст ошибки


def mark_date_done(completed_set, date_str):                  # два параметра: множество дат и одна дата-строка
    completed_set.add(date_str)                               # .add — добавить дату в множество (в множестве не бывает дублей)
    save_checkpoint(completed_set)                            # сразу записать обновлённое множество на Drive (вызов функции выше)


# ---------- АВТО-ОБНОВЛЕНИЕ _token ----------

def refresh_token():
    try:                                                      # сеть может не ответить — оборачиваем в try
        r = session.get("https://nrat.ukrintei.ua/searchdb/", timeout=30)   # session.get — запросить страницу; timeout=30 — ждать ответа не дольше 30 сек; r — ответ сервера
        soup = BeautifulSoup(r.content, 'html.parser')        # BeautifulSoup разбирает HTML-страницу в удобный для поиска объект soup
        tok = soup.find('input', attrs={'name': '_token'})    # ищем на странице тег <input name="_token"> — в нём лежит токен
        if tok and tok.get('value'):                          # if ... and ... — если тег НАЙДЕН И у него есть атрибут value
            BASE_PARAMS['_token'] = tok['value']              # записываем свежий токен в общий словарь параметров (используется во всех запросах)
            print("  🔑 _token обновлён")                     # сообщение в лог, что всё получилось
            return True                                       # вернуть True = «успех» и выйти из функции
    except Exception as e:                                    # если запрос упал
        print(f"  ⚠️ Не удалось обновить _token ({e}), используем прежний")   # предупреждаем и продолжаем со старым токеном
    return False                                              # сюда попадаем, если токен не обновился — вернуть False = «неудача»


# ---------- ДИАПАЗОНЫ ДАТ ----------

def generate_date_ranges(start_date_str, end_date_str, days_per_chunk):   # три параметра: начало, конец, размер отрезка в днях
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")   # strptime превращает текст "2025-01-01" в настоящую дату, с которой можно считать
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")       # то же для конечной даты
    ranges = []                                               # пустой список [], сюда будем складывать отрезки
    cur = start_date                                          # cur — «текущая» дата, с которой шагаем; стартуем с начала
    while cur <= end_date:                                    # while = «пока»: повторять, ПОКА текущая дата не перешла конец
        cur_end = min(cur + timedelta(days=days_per_chunk - 1), end_date)   # конец отрезка = старт + (N-1) дней; min(...) не даёт залезть за end_date
        ranges.append((cur.strftime("%Y-%m-%d"), cur_end.strftime("%Y-%m-%d")))   # .append добавляет в список пару (начало, конец); strftime превращает дату обратно в текст
        cur = cur_end + timedelta(days=1)                     # сдвигаем cur на следующий день после конца отрезка
    return ranges                                             # вернуть готовый список всех отрезков (дней)


def build_search_url(date_from, date_to, page=1):             # page=1 — значение ПО УМОЛЧАНИЮ: если не передать page, будет 1
    params = BASE_PARAMS.copy()                               # .copy() делает КОПИЮ общего словаря, чтобы менять её, не трогая оригинал
    params['dateFromSearch'] = date_from                      # кладём в копию дату начала поиска
    params['dateToSearch'] = date_to                          # и дату конца поиска
    params['pa'] = str(page)                                  # ВАЖНО: параметр страницы называется 'pa', НЕ 'page'! str(...) превращает число в текст
    return BASE_SEARCH_URL + urlencode(params, doseq=True)    # urlencode склеивает словарь в строку ?ключ=значение&...; + приклеивает её к адресу сайта


# ---------- ЧИСЛО НАЙДЕННЫХ ДОКУМЕНТОВ ----------

def extract_total_results(soup):                              # soup — уже разобранная страница (объект BeautifulSoup)
    try:                                                      # разбор HTML может не найти нужного — на всякий случай try
        page_info = soup.find('div', class_='page_info')      # ищем блок <div class="page_info"> — там обычно стоит счётчик документов
        if page_info:                                         # если такой блок нашёлся
            m = re.search(r'Знайдено документів:\s*(\d+)', page_info.get_text(strip=True))   # re.search ищет по шаблону; \s* — пробелы, (\d+) — поймать число; get_text — взять текст блока
            if m:                                             # если шаблон совпал (число найдено)
                return int(m.group(1))                        # m.group(1) — пойманное число (текстом), int(...) делает из него число; вернуть его
        m = re.search(r'Знайдено документів:\s*(\d+)', soup.get_text(" ", strip=True))   # запасной вариант: ищем ту же фразу во ВСЁМ тексте страницы
        if m:                                                 # если нашли
            return int(m.group(1))                            # вернуть число
    except Exception:                                         # любая ошибка разбора
        pass                                                  # pass = «ничего не делать», просто идём дальше
    return 0                                                  # ничего не нашли — считаем, что документов 0


# ---------- ПАРСИНГ СТРАНИЦЫ ВЫДАЧИ (с повторами при таймауте) ----------

def get_search_results_page(url, retry_count=3):              # url — адрес страницы; retry_count=3 — сколько раз пробовать при сбое (по умолч. 3)
    for attempt in range(retry_count):                        # for ... in range(3) — повторить тело до 3 раз; attempt = номер попытки (0,1,2)
        try:                                                  # пробуем загрузить страницу; при сбое уйдём в except ниже
            response = session.get(url, timeout=60)           # запрашиваем страницу; ждём ответа максимум 60 сек; response — ответ сервера
            response.raise_for_status()                       # если сервер вернул ошибку (например 500) — поднять исключение, уйти в except
            soup = BeautifulSoup(response.content, 'html.parser')   # разбираем HTML в удобный объект soup
            results = []                                      # пустой список — сюда сложим найденные документы
            for card in soup.find_all('div', class_='my-card-body'):   # find_all находит ВСЕ карточки документов; перебираем каждую как card
                link_tag = card.find('a', target='_blank')    # внутри карточки ищем ссылку <a target="_blank"> — это ссылка на документ
                if link_tag and link_tag.get('href'):         # если ссылка есть И у неё есть адрес (href)
                    reg_number = link_tag.get_text(strip=True)   # текст ссылки = регистрационный номер; strip=True убирает лишние пробелы
                    detail_url = link_tag['href']             # сам адрес страницы документа
                    card_text = card.get_text(separator=' ', strip=True)   # весь текст карточки одной строкой (через пробел)
                    parts = card_text.split('Керівник:')      # .split разрезает текст по слову 'Керівник:' — название работы стоит до него
                    title = parts[0].strip() if len(parts) > 1 else card_text[:100].strip()   # если разрез удался — берём кусок до 'Керівник:'; иначе первые 100 символов (короткая запись if/else в одну строку)
                    doc_id = detail_url.rstrip('/').split('/')[-1]   # из адреса берём последний кусок после '/' — это числовой ID; rstrip('/') убирает '/' на конце
                    pdf_url = f"https://dir.ukrintei.ua/view/ok/{doc_id}"   # собираем прямую ссылку на PDF, подставляя {doc_id} в f-строку
                    results.append({                          # добавляем в список словарь {ключ: значение} со всеми полями документа
                        'registration': reg_number, 'detail_url': detail_url,
                        'pdf_url': pdf_url, 'title': title, 'doc_id': doc_id
                    })
            return results, soup                              # успех: возвращаем СРАЗУ ДВА значения — список документов и разобранную страницу
        except Exception as e:                                # сюда попадаем при сбое сети/сервера
            print(f"    ⚠️ загрузка выдачи, попытка {attempt + 1}/{retry_count}: {e}")   # сообщаем, какая попытка не удалась (attempt+1, т.к. отсчёт с 0) и текст ошибки
            if attempt < retry_count - 1:                     # если это НЕ последняя попытка
                time.sleep(5)                                 # подождать 5 сек и цикл for попробует снова
    return None, None                                         # все попытки провалились — возвращаем None, None (= ошибка сети, день не готов)


# ---------- СКАЧИВАНИЕ PDF ----------

def download_pdf(pdf_url, registration, title, doc_id, folder, retry_count=2):   # параметры: ссылка на PDF, рег.номер, название, ID, папка дня, число попыток (по умолч. 2)
    safe_reg = re.sub(r'[^\w\-_.]', '_', registration)        # re.sub заменяет все «опасные» для имени файла символы (всё, кроме букв/цифр/-_. ) на '_'
    filename = re.sub(r'_+', '_', f"{safe_reg}_{doc_id}.pdf") # собираем имя «рег_ID.pdf»; вторым re.sub схлопываем несколько '_' подряд в один
    filepath = os.path.join(folder, filename)                 # склеиваем путь папки и имя файла в полный путь (os.path.join сам ставит правильный слэш)

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:   # если файл УЖЕ есть И его размер больше 0 байт
        return 'skip', filepath                               # значит уже качали — возвращаем 'skip' и путь, ничего не делаем

    for attempt in range(retry_count):                        # повторяем скачивание до retry_count раз при сбоях
        try:
            response = session.get(pdf_url, stream=True, timeout=60)   # запрашиваем PDF; stream=True — качать по кусочкам (для больших файлов); timeout=60
            if response.status_code == 404:                   # 404 = сервер говорит «такого файла нет»
                return 'notpdf', None                         # у записи просто нет PDF — возвращаем 'notpdf', повторять не нужно
            response.raise_for_status()                       # другая ошибка сервера (500 и т.п.) — поднять исключение, уйти в except
            with open(filepath, 'wb') as f:                   # открываем файл на запись в ДВОИЧНОМ режиме ('wb' = write binary), т.к. PDF не текст
                for chunk in response.iter_content(chunk_size=8192):   # читаем ответ кусками по 8192 байта
                    if chunk:                                 # если кусок не пустой
                        f.write(chunk)                        # дописываем его в файл
            if os.path.getsize(filepath) > 0:                 # файл записан — проверяем, что он не нулевого размера
                with open(filepath, 'rb') as f:               # открываем уже записанный файл на чтение в двоичном режиме ('rb')
                    if f.read(4).startswith(b'%PDF'):         # читаем первые 4 байта; настоящий PDF всегда начинается с %PDF (b'...' — байты, не текст)
                        return 'ok', filepath                 # это валидный PDF — успех, возвращаем 'ok' и путь
                os.remove(filepath)                           # сюда дошли — файл НЕ PDF (например HTML-заглушка); удаляем мусор
                return 'notpdf', None                         # и возвращаем 'notpdf'
            else:                                             # размер файла == 0
                os.remove(filepath)                           # удаляем пустышку
                return 'empty', None                          # возвращаем 'empty'
        except Exception as e:                                # сбой сети при скачивании
            print(f"    ❌ Ошибка скачивания: {e}")           # показываем текст ошибки
            if attempt < retry_count - 1:                     # если не последняя попытка
                time.sleep(3)                                 # пауза 3 сек и пробуем снова
    return 'fail', None                                       # все попытки не удались — возвращаем 'fail'

print("✅ Функции загружены")                                 # печатается в самом конце — сигнал, что все функции выше успешно определены


In [ ]:
# ==============================================
# ЯЧЕЙКА 4 — ОСНОВНАЯ ЛОГИКА
# ==============================================

def scrape_day(date_from, date_to, day_folder):
    """
    Скачивает ВСЕ PDF за один день — обходит все страницы пагинации.

    Алгоритм:
      1. Запрашивает первую страницу результатов за указанный день.
      2. Из первой страницы определяет total_results и вычисляет
         количество страниц: total_pages = ceil(total_results / 10).
      3. Последовательно обходит все страницы, скачивая PDF с каждой.
      4. Дедуплицирует по doc_id (seen_ids), чтобы не скачивать одно
         и то же дважды, если сайт вернул дубли на соседних страницах.
      5. Останавливается когда:
         - прошли все страницы (page >= total_pages), или
         - очередная страница вернула пустой список, или
         - достигнут предел page >= 85 (страховка от бесконечного цикла).

    Флаг day_ok:
      True  — день обработан успешно, можно помечать в чекпоинте.
      False — страница выдачи не загрузилась из-за ошибки сети.
              День НЕ помечается готовым и будет повторён при следующем запуске.

    Параметры:
        date_from  — строка 'YYYY-MM-DD', начало диапазона (обычно = date_to)
        date_to    — строка 'YYYY-MM-DD', конец диапазона
        day_folder — полный путь к папке дня на Drive, куда сохранять PDF

    Возвращает: кортеж (stats, day_ok)
        stats  — словарь {'ok': N, 'skip': N, 'fail': N, 'notpdf': N}
        day_ok — True если день обработан успешно, False при ошибке сети
    """
    os.makedirs(day_folder, exist_ok=True)
    page = 1
    page_size = None
    total_pages = None
    seen_ids = set()
    stats = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    day_ok = True

    while True:
        url = build_search_url(date_from, date_to, page)
        results, soup = get_search_results_page(url)

        if results is None:           # сеть не ответила после всех попыток
            print("    ⚠️ страница выдачи не загрузилась — день будет повторён позже")
            day_ok = False
            break

        if page == 1:
            total_results = extract_total_results(soup)
            page_size = len(results) if results else 10
            total_pages = ((total_results + 9) // 10) if total_results else None
            if total_results:
                print(f"    найдено {total_results} рез., {total_pages} стр.")
            if total_results >= 1000:
                print("    ⚠️ ВНИМАНИЕ: 1000+ результатов за день — возможно, не все попадут в выдачу")

        if not results:               # страница загрузилась, но документов нет
            if page == 1:
                print("    нет результатов")
            break

        new_results = [r for r in results if r['doc_id'] not in seen_ids]
        if page > 1 and not new_results:
            break
        for r in new_results:
            seen_ids.add(r['doc_id'])

        for i, r in enumerate(new_results, 1):
            status, _ = download_pdf(r['pdf_url'], r['registration'], r['title'], r['doc_id'], day_folder)
            stats[status if status in stats else 'fail'] += 1
            if i < len(new_results):
                time.sleep(DELAY_BETWEEN_FILES)

        if total_pages is not None:
            go_next = page < total_pages
        else:
            go_next = bool(page_size and len(results) >= page_size)
        if not go_next or page >= 85:
            break

        page += 1
        time.sleep(DELAY_BETWEEN_PAGES)

    return stats, day_ok


def run_year(year):
    """
    Собирает все PDF за целый год, с продолжением после обрыва.

    Алгоритм:
      1. Загружает чекпоинт — список дней, уже успешно обработанных ранее.
      2. Обновляет _token перед стартом (refresh_token).
      3. Генерирует список всех дней года от 01-01 до 31-12.
      4. Для каждого дня:
         - если день уже в чекпоинте — пропускает (continue).
         - вызывает scrape_day() для скачивания.
         - если день прошёл успешно — добавляет в чекпоинт и сохраняет на Drive.
         - если ошибка сети — добавляет в список incomplete (будет повторён).
         - делает паузу DELAY_BETWEEN_DAYS секунд перед следующим днём.
      5. Выводит итоговую статистику.

    При повторном запуске (после обрыва) уже завершённые дни пропускаются
    благодаря чекпоинту, и сбор продолжается с первого незавершённого дня.

    Параметры:
        year — числовой год (int), например 2025

    Возвращает: словарь grand со статистикой за весь год
        {'ok': N, 'skip': N, 'fail': N, 'notpdf': N}
    """
    completed = load_checkpoint()
    print(f"📌 По году {year} уже завершено дней: {len(completed)}")
    refresh_token()

    date_ranges = generate_date_ranges(f"{year}-01-01", f"{year}-12-31", DAYS_PER_CHUNK)
    grand = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    incomplete = []

    print("\n" + "=" * 70)
    print(f"ГОД {year} — {len(date_ranges)} дней")
    print("=" * 70)

    for idx, (date_from, date_to) in enumerate(date_ranges, 1):
        if date_from in completed:
            continue

        print(f"\n📅 {date_from}  ({idx}/{len(date_ranges)})")
        stats, day_ok = scrape_day(date_from, date_to, day_folder_for(year, date_from))
        for k in grand:
            grand[k] += stats[k]
        print(f"    итог дня: ✅{stats['ok']} ⏭{stats['skip']} 📄✗{stats['notpdf']} ❌{stats['fail']}"
              f"   |   ВСЕГО за год ✅{grand['ok']}")

        if day_ok:
            mark_date_done(completed, date_from)   # помечаем готовым ТОЛЬКО при успехе
        else:
            incomplete.append(date_from)
        time.sleep(DELAY_BETWEEN_DAYS)

    print("\n" + "=" * 70)
    print(f"ГОД {year} ГОТОВ!")
    print(f"Скачано новых PDF: {grand['ok']} | пропущено (уже было): {grand['skip']} | "
          f"без файла: {grand['notpdf']} | ошибок: {grand['fail']}")
    if incomplete:
        print(f"⚠️ Дней с ошибкой сети (НЕ завершены, запусти ячейку «весь год» ещё раз — доберутся): {len(incomplete)}")
        print("   ", incomplete)
    print("=" * 70)
    return grand

print("✅ Функции и логика загружены. Запускай ячейку «СОБРАТЬ ВЕСЬ ГОД».")

In [ ]:
# ==============================================
# СОБРАТЬ ВЕСЬ ГОД ЦЕЛИКОМ  (⚠️ идёт ЧАСЫ)
# Собирает год из YEAR (ячейка 2).
# Если Colab оборвался — выполни ячейки 1–4 и эту снова: продолжит с места обрыва.
# ==============================================
year_stats = run_year(YEAR)